# 13 - Uncertainty Budget: How Far to Trust a Species Record

Notebooks 01 to 12 treated the database as if every record deserved the same
trust. This one asks how far a single species can deviate from reference
thermochemistry. Since 0.5.0, `pyglenn` ships a per-species error envelope
exposed by `ThermochemicalCalculator.uncertainty`. The envelope summarises, in a
single temperature-independent set of numbers, the difference between the
bundled NASA polynomials and NIST-JANAF tables: a bias and a two-sigma band for
Cp and H, and an absolute two-sigma band for S.

The plan is short. We start with what the method returns, then measure how much
of the catalogue is actually validated, look at the species the envelope flags,
and finish by propagating the band into a reaction enthalpy.

In [1]:
import pandas as pd

from pyglenn import ThermochemicalCalculator

pd.set_option("display.float_format", lambda v: f"{v:,.3f}")


## 1. What the method returns

`uncertainty(species_id, temperature)` returns a dictionary. The temperature is
checked against the species range but does not change the numbers, because the
envelope is one summary for the whole interval.

| key | meaning |
|-----|---------|
| `cp_rel_bias` | mean relative deviation of Cp, in percent |
| `cp_rel_2sigma` | two-sigma relative band of Cp, in percent |
| `h_rel_bias` | mean relative deviation of the standardized enthalpy, in percent |
| `h_rel_2sigma` | two-sigma relative band of H, in percent |
| `s_abs_2sigma` | two-sigma band of S, absolute, in J/(mol K) |
| `n_ref_points` | number of reference points behind the envelope |
| `source` | `nist-janaf` for validated species, `extrapolated` otherwise |

In [2]:
with ThermochemicalCalculator() as calc:
    co2 = calc.get_available_species("CO2", exact_match=True)[0]
    envelope = calc.uncertainty(co2.id, 1000.0)

for key, value in envelope.items():
    print(f"{key:15s}: {value}")

species_id     : 316
species_name   : CO2
temperature    : 1000.0
cp_rel_bias    : 0.1874
cp_rel_2sigma  : 1.4138
h_rel_bias     : 0.0221
h_rel_2sigma   : 0.1668
s_abs_2sigma   : 0.0912
n_ref_points   : 16
source         : nist-janaf


## 2. How much of the catalogue is validated

The label in `source` matters more than the numbers. A validated species was
compared point by point against tabulated reference data. Everything else
carries a default band that came from the validation study, not from a fit to
that particular species.

In [3]:
with ThermochemicalCalculator() as calc:
    catalogue = calc.get_available_species("")
    sources = {}
    for s in catalogue:
        low, high = calc.db.get_temperature_bounds(s.id)
        env = calc.uncertainty(s.id, 0.5 * (low + high))
        label = "missing" if env is None else env["source"]
        sources[label] = sources.get(label, 0) + 1

for label, count in sources.items():
    print(f"{label:12s}: {count:5d}")

extrapolated:  2017
nist-janaf  :    18


The imbalance is large. Only eighteen species were validated by hand, and the
remaining two thousand and seventeen inherit the default envelope. That is not a defect of
the library, it is a statement about how much work validation costs, and it
should temper any claim about accuracy made for an arbitrary species.

## 3. Species flagged by the envelope

The envelope earns its keep as a screening device. Among the validated species
used in this labbook, carbon dioxide stays at 1.4 percent on Cp, oxygen at 0.8
and nitrogen at 0.2; water reaches 3.1, and extrapolated entries carry the
default band of about 9 percent.

In [4]:
names = ["CO2", "H2O", "O2", "N2", "CH4", "C2H5OH"]

rows = []
with ThermochemicalCalculator() as calc:
    for name in names:
        s = calc.get_available_species(name, exact_match=True)[0]
        env = calc.uncertainty(s.id, 1000.0)
        rows.append({
            "species": s.name,
            "source": env["source"],
            "Cp bias [%]": env["cp_rel_bias"],
            "Cp 2sigma [%]": env["cp_rel_2sigma"],
            "H 2sigma [%]": env["h_rel_2sigma"],
            "S 2sigma [J/mol/K]": env["s_abs_2sigma"],
        })

pd.DataFrame(rows).set_index("species").round(2)

,source,Cp bias [%],Cp 2sigma [%],H 2sigma [%],S 2sigma [J/mol/K]
species,,,,,
CO2,nist-janaf,0.190,1.410,0.170,0.090
H2O,nist-janaf,1.100,3.100,1.630,1.170
O2,nist-janaf,0.100,0.770,0.280,0.150
N2,nist-janaf,0.030,0.180,0.020,0.010
CH4,nist-janaf,8.820,27.610,13.550,17.290
C2H5OH,extrapolated,0.000,9.070,4.300,5.100


Methane is the exception, at 27.6 percent, one order of magnitude above the
others.
A flag of that size does not prove that the record is wrong, but it does
establish where to look first. The next section follows that lead.

## 4. Following up on a flagged species

The first question for a flagged record is whether the fundamental anchors hold.
Three values are enough: the heat capacity, the entropy and the enthalpy of
formation at 298.15 K. Reference values for methane are widely tabulated, and
the comparison takes one loop.

In [5]:
anchors = {                      # NIST reference values for CH4(g)
    "Cp(298.15 K) [J/mol/K]": 35.69,
    "S(298.15 K) [J/mol/K]": 186.25,
    "dHf(298.15 K) [kJ/mol]": -74.87,
}

with ThermochemicalCalculator() as calc:
    ch4 = calc.get_available_species("CH4", exact_match=True)[0]
    props = calc.calculate_properties(ch4.id, 298.15)
    computed = {
        "Cp(298.15 K) [J/mol/K]": props.cp,
        "S(298.15 K) [J/mol/K]": props.s,
        "dHf(298.15 K) [kJ/mol]": props.h_relative / 1000.0,
    }

for label, reference in anchors.items():
    print(f"{label:24s} pyglenn {computed[label]:9.3f}   NIST {reference:9.3f}")

Cp(298.15 K) [J/mol/K]   pyglenn    35.691   NIST    35.690
S(298.15 K) [J/mol/K]    pyglenn   186.371   NIST   186.250
dHf(298.15 K) [kJ/mol]   pyglenn   -74.600   NIST   -74.870


All three anchors land within a few tenths of a percent of the reference, so
the record is sound near ambient temperature. The flagged band therefore
concentrates at high temperature and in the entropy and enthalpy offsets, not in
the polynomial shape we use most. The lesson is procedural: an envelope is a
pointer, and the pointer has to be followed with an independent check before
anything is declared broken.

## 5. Propagating the band into a reaction enthalpy

For a linear combination such as a reaction enthalpy, the bands combine in
quadrature with the stoichiometric coefficients as weights. The example below
takes the complete combustion of methane at 298.15 K and treats the relative
band of H as if it applied to the standardized enthalpy of each species.

In [6]:
reaction = {"CH4": -1.0, "O2": -2.0, "CO2": +1.0, "H2O": +2.0}

terms = []
with ThermochemicalCalculator() as calc:
    for name, nu in reaction.items():
        s = calc.get_available_species(name, exact_match=True)[0]
        h = calc.calculate_properties(s.id, 298.15).h_relative / 1000.0
        env = calc.uncertainty(s.id, 298.15)
        terms.append({
            "species": name,
            "nu": nu,
            "H [kJ/mol]": h,
            "2sigma [kJ/mol]": abs(h) * env["h_rel_2sigma"] / 100.0,
        })

table = pd.DataFrame(terms).set_index("species")
dH = float((table["nu"] * table["H [kJ/mol]"]).sum())
band = float(((table["nu"] * table["2sigma [kJ/mol]"]) ** 2).sum() ** 0.5)

print(table.round(2))
print(f"\nDelta H_rxn(298.15 K) = {dH:8.1f} kJ/mol")
print(f"combined 2sigma       = {band:8.1f} kJ/mol  ({100 * band / abs(dH):.1f} % of the enthalpy)")

            nu  H [kJ/mol]  2sigma [kJ/mol]
species                                    
CH4     -1.000     -74.600           10.110
O2      -2.000      -0.000            0.000
CO2      1.000    -393.510            0.660
H2O      2.000    -241.830            3.940

Delta H_rxn(298.15 K) =   -802.6 kJ/mol
combined 2sigma       =     12.8 kJ/mol  (1.6 % of the enthalpy)


The combined band of roughly 13 kJ/mol corresponds to 1.6 percent of the
reaction enthalpy, and almost all of it comes from the fuel. This asymmetry is
typical: the species with the largest formation enthalpy dominates the budget,
so a validated fuel is worth more than a validated product.

## 6. What to do with it

Check the source label before trusting a species that drives a conclusion. When
the record is validated, quote the band next to the computed value. When it is
extrapolated, say so. When a band looks anomalous, compare the species against
an independent table before using it in a quantitative model. The envelope
narrows the search, it does not replace the reference.